In [1]:
%load_ext autoreload
%autoreload 2
import pathlib

import numpy as np
import toml
import matplotlib.pyplot as plt
import pandas as pd
import pyerrors as pe
import h5py

from compute_observables import (
        compute_average_magnetisation,
        compute_average_energy,
        compute_susceptibility,
        compute_specific_heat,
        compute_binder_cumulant,
    )
from utils.h5_utils import (
    import_lattice,
)
from lattice_plots import (
    _generate_lattice
)

# Matplotlib plot parameters
plt.rcParams['text.usetex'] = True
plt.rcParams.update({'font.size':14, 'figure.autolayout':True})



project_name = "temperature30_0-3_1-5_l100_dim2_long"
project_path = pathlib.Path("/home/alvaro/Documents/trinity/year4/capstone/capstone-code/projects") / project_name 

def import_observable(directory: pathlib.Path, observable_name: str)->int | float:
    f = h5py.File(directory / "results.h5", "r")    
    observable = np.array(f["observables"][observable_name])
    f.close()
    return observable


In [19]:
from matplotlib import colors
project_name = "above_critical_temp_test"
project_root = pathlib.Path("/home/alvaro/Documents/trinity/year4/capstone/capstone-code/projects") 
# project_root = pathlib.Path("/home/users/romeroca/capstone-code/projects") 
parameter_combination = 0
project_path = project_root/ project_name / f"parameter-config-{parameter_combination}"
config = toml.load(project_path / "config.toml")
time = 20
lattice = import_lattice(project_path, time)

fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()

angles = _generate_lattice(time,lattice, config)
y,x = np.indices(angles.shape)

u = np.cos(angles)
v = np.sin(angles)
im = ax2.imshow(
    _generate_lattice(time, import_lattice(project_path,time), config),
      cmap="hsv",
      vmin = 0,
      vmax = 2*np.pi,
      interpolation='bilinear',
      origin = "lower")
norm = colors.Normalize(vmin=0, vmax=2*np.pi)

# q = ax.quiver(x, y, u, v, angles, cmap="hsv", pivot="mid", scale = 30, norm = norm)
step = 1
q_black = ax2.quiver(
    # x[::step, ::step],
    x,
    # y[::step, ::step],
    y,
    # u[::step, ::step],
    u,
    # v[::step, ::step],
    v,
    color="black",
    pivot="mid",
    # scale=50,
    alpha=0.7,
    width=0.003,
    headwidth=3,
    headlength=4,
    headaxislength=3.5,
)
q_chat = ax.quiver(
    x[::step, ::step],
    y[::step, ::step],
    u[::step, ::step],
    v[::step, ::step],
    angles[::step, ::step],
    cmap="hsv",
    norm=norm,
    pivot="mid",
    scale=25,
    width=0.003,
    headwidth=3,
    headlength=4,
    headaxislength=3.5,
)


ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
#q = ax.quiver(x[::step], y[::step], u[::step], v[::step], angles[::step], cmap="hsv", pivot="mid", scale = 20, norm = norm)
cbar = fig.colorbar(q_chat, ax=ax, label="Spin angle (radians)")
cbar.set_ticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
cbar.set_ticklabels(["0", r"$\pi$/2", r"$\pi$", r"3$\pi$/2", r"2$\pi$"])
cbar2 = fig2.colorbar(im, ax=ax2, label="Spin angle (radians)")
cbar2.set_ticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
cbar2.set_ticklabels(["0", r"$\pi$/2", r"$\pi$", r"3$\pi$/2", r"2$\pi$"])

ax.set_aspect('equal')
ax2.set_aspect('equal')
ax.set_title(f"XY model for $J = 1$, $T = {config['physical_settings']['temperature']} / k_B$")
ax2.set_title(f"XY model for $J = 1$, $T = {config['physical_settings']['temperature']} / k_B$")
fig.savefig("color_arrow_plot.pdf")
fig2.savefig("lattice_arrow_plot.pdf")

In [13]:
# update the plot and make an animation
config

{'git_hash': 'v0.0-60-g49aab11-dirty',
 'seed': 0,
 'physical_settings': {'temperature': 0.3, 'L': 50, 'dimension': 2, 'J': 1},
 'simulation_settings': {'total_sweeps': 10000, 'rg_method': False}}